# Laya full fine-tune (Kaggle / Colab, free GPU)

The encoder (322M) plus AdamW states does not fit in 4 GB of VRAM, so the local run trains only the head (B3 frozen: macro-F1 0.41 vs B1 0.69 on test). Here the encoder is trained too.

**Before running:** upload `training/data/laya/train.jsonl` and `val.jsonl` (produced by `python training/laya_finetune.py --limit 1`) as a Kaggle Dataset or to Colab `/content/data/`. GPU: T4 or P100 (Settings → Accelerator).

In [ ]:
!pip -q install laya

In [ ]:
import json
import os
import time

DATA = next(p for p in ('/kaggle/input/laya-data', '/content/data', 'data') if os.path.exists(p))
print(DATA, os.listdir(DATA))

In [ ]:
import torch
from laya.train import TrainConfig, finetune

CHECKPOINT = 'convaiinnovations/laya-multilingual'
config = TrainConfig(epochs=3, micro_batch=8, grad_accum=4, freeze_encoder=False,
                     shuffle_options=('choice',), max_len=320, eval_data=f'{DATA}/val.jsonl')
started = time.time()
summary = finetune(f'{DATA}/train.jsonl', CHECKPOINT, 'laya_ft', config,
                   device='cuda' if torch.cuda.is_available() else 'cpu')
summary['train_seconds'] = round(time.time() - started, 1)
print(json.dumps(summary['train_report']['comparison'], indent=2))

In [ ]:
!zip -qr laya_ft.zip laya_ft && ls -lh laya_ft.zip

## Import

Download `laya_ft.zip`, unzip it into `models/laya_ft/` in the repo, then rerun locally:

```bash
uv run --no-sync python training/calibrate.py --checkpoint models/laya_ft --device cuda
LAYA_DEVICE=cuda uv run --no-sync python eval/run_eval.py --router b3 --set test --policy --limit 600
```

Acceptance (PLAN §10, M3): B3 macro-F1 ≥ B1, ECE < 0.05 after calibration.